# Esquema: clasificación multiclase (MVP manual)

Target **texto** → 0..K-1 manual. Misma idea: un pipeline por modelo, tabla de **accuracy**.

Siguiente: [07.b multiclase](../07.b-ejemplos-supervisados/03-clasificacion-multiple.ipynb).


## 1. CSV, tipos y faltantes

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report

df = pd.read_csv("data/datos_flores.csv")
print(df.dtypes)
print("\nFaltantes:\n", df.isna().sum())


sepal_length    float64
sepal_width     float64
especie          object
dtype: object

Faltantes:
 sepal_length    1
sepal_width     1
especie         1
dtype: int64


## 2. Target multiclase → 0..K-1

In [2]:
df = df.dropna(subset=["especie"]).copy()
ORDEN_CLASES = ["setosa", "versicolor", "virginica"]
MAPA_MULTI = {n: i for i, n in enumerate(ORDEN_CLASES)}
y = df["especie"].str.strip().str.lower().map(MAPA_MULTI).astype(int)


## 3. Features numéricas

In [3]:
cols_num = ["sepal_length", "sepal_width"]
X = df[cols_num].astype(float).copy()
for col in cols_num:
    X[col] = X[col].fillna(X[col].median())


## 4. Split estratificado

In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)


## 5. Varios modelos en Pipeline

In [5]:
def build_models(n_classes):
    """Misma lista que 07.b (comenta entradas para excluir modelos)."""
    from sklearn.ensemble import (
        GradientBoostingClassifier,
        HistGradientBoostingClassifier,
        RandomForestClassifier,
    )
    from sklearn.linear_model import LogisticRegression, SGDClassifier
    from sklearn.multiclass import OneVsOneClassifier, OneVsRestClassifier
    from sklearn.neighbors import KNeighborsClassifier
    from sklearn.svm import SVC
    from sklearn.tree import DecisionTreeClassifier
    from xgboost import XGBClassifier
    from catboost import CatBoostClassifier

    if n_classes < 3:
        raise ValueError(f"build_models: K={n_classes} < 3 (multiclase)")
    return {
        "LogisticRegression": LogisticRegression(max_iter=2000, random_state=RANDOM_STATE),
        "SGDClassifier": SGDClassifier(
            loss="log_loss",
            max_iter=2000,
            tol=1e-3,
            random_state=RANDOM_STATE,
        ),
        "SVC": SVC(random_state=RANDOM_STATE),
        "OneVsOneClassifier": OneVsOneClassifier(SVC(random_state=RANDOM_STATE)),
        "OneVsRestClassifier": OneVsRestClassifier(SVC(random_state=RANDOM_STATE)),
        "KNN": KNeighborsClassifier(n_neighbors=5, n_jobs=-1),
        "DecisionTree": DecisionTreeClassifier(
            criterion="gini",
            splitter="best",
            max_depth=None,
            min_samples_split=2,
            min_samples_leaf=1,
            min_weight_fraction_leaf=0.0,
            max_features=None,
            max_leaf_nodes=None,
            min_impurity_decrease=0.0,
            random_state=RANDOM_STATE,
        ),
        "RandomForest": RandomForestClassifier(
            n_estimators=100,
            criterion="gini",
            max_depth=None,
            min_samples_split=2,
            min_samples_leaf=1,
            min_weight_fraction_leaf=0.0,
            max_features="sqrt",
            max_leaf_nodes=None,
            min_impurity_decrease=0.0,
            bootstrap=True,
            oob_score=False,
            max_samples=None,
            random_state=RANDOM_STATE,
            n_jobs=-1,
        ),
        "GradientBoosting": GradientBoostingClassifier(random_state=RANDOM_STATE),
        "HistGradientBoosting": HistGradientBoostingClassifier(random_state=RANDOM_STATE),
        "XGBoost": XGBClassifier(
            random_state=RANDOM_STATE,
            verbosity=0,
            n_estimators=100,
            objective="multi:softmax",
            num_class=n_classes,
            n_jobs=-1,
        ),
        "CatBoost": CatBoostClassifier(
            random_state=RANDOM_STATE,
            verbose=False,
            iterations=100,
            allow_writing_files=False,
            loss_function="MultiClass",
        ),
    }


RANDOM_STATE = 42
N_CLASSES = int(y.nunique())
MODELS = build_models(N_CLASSES)


filas = []
mejor_nombre, mejor_acc, mejor_pred = None, -1.0, None

for nombre, modelo in MODELS.items():
    pipe = make_pipeline(StandardScaler(), modelo)
    pipe.fit(X_train, y_train)
    pred = pipe.predict(X_test)
    acc = accuracy_score(y_test, pred)
    filas.append({"modelo": nombre, "accuracy": acc})
    if acc > mejor_acc:
        mejor_acc, mejor_nombre, mejor_pred = acc, nombre, pred

display(pd.DataFrame(filas).sort_values("accuracy", ascending=False).round(4))

print(f"\nMejor accuracy: {mejor_nombre} ({mejor_acc:.2f})")
print(classification_report(y_test, mejor_pred, target_names=ORDEN_CLASES))


,modelo,accuracy
0,LogisticRegression,0.5000
1,KNN,0.5000
2,DecisionTree,0.3333
3,RandomForest,0.3333
4,SVC,0.3333



Mejor accuracy: LogisticRegression (0.50)
              precision    recall  f1-score   support

      setosa       1.00      1.00      1.00         2
  versicolor       0.00      0.00      0.00         2
   virginica       0.33      0.50      0.40         2

    accuracy                           0.50         6
   macro avg       0.44      0.50      0.47         6
weighted avg       0.44      0.50      0.47         6

